# Permian Basin Flaring Site Catalog & Well Site-ID Assignment

**Pipeline stage:** Step 4 of the Texas RRC oil production data pipeline (follows the Permian Basin
subsetting notebook, which produces the Permian-only well and production CSVs this notebook reads).

This notebook has two parts:

1. **Flaring/combustion site catalog** — reads a satellite-derived, worldwide catalog of persistent
   high-temperature combustion sites (VIIRS Nightfire), restricts it to the Permian Basin, and saves
   a clean table of site polygons.
2. **Well → site assignment** — spatially matches every Permian well against those site polygons
   (a point-in-polygon test: does this well's location fall inside a known flaring/combustion site?)
   and tags each well with the `site_id` it falls inside, or `-1` if it doesn't fall inside any.

Every output of this notebook is written to the same `permian_only` folder used by the previous
pipeline stage, so all Permian-Basin-scoped artifacts live in one place.

## 0. Overview

### What VIIRS Nightfire detects

The VIIRS Nightfire (VNF) satellite instrument detects high-temperature combustion sources at night,
worldwide — gas flares, refinery/industrial furnaces, coal seam fires, volcanoes, and more. Over many
years of repeated satellite passes, detections that consistently recur at the same physical location
get clustered into a single polygon ("bubble") representing that persistent combustion site, and each
bubble is classified into a site type: `upstream` (oil & gas production sites — the category of
interest here), `downstream` (refineries etc.), `industrial`, `landfill`, `coal_mine`, `cement`,
`metallurgy`, `sawmill`, `volcano`, or `unknown`/`unique` for sites that don't cleanly fit elsewhere.

### How each site's ID is determined

Every polygon in the catalog carries two identifiers:

- `id` — a long descriptive string assigned by the detection pipeline (e.g.
  `IR_2012-2020_tile_-105,30,-90,45_feature_1844_source_0`), encoding the processing tile and feature
  number. This is kept below as `catalog_id`.
- `index` — a plain integer giving that polygon's row position within its category's complete
  worldwide attribute table.

This notebook uses `index`, renamed to `site_id`, as the unique identifier for each site going
forward. It isn't invented here — it's the identifier the data provider already assigned to that
polygon within this catalog release, so reusing it means these site IDs stay consistent with anything
else derived from the same release. Because `index` is only guaranteed unique *within* a single
category's own worldwide table (not across categories), Section 5 runs an explicit sanity check
confirming `site_id` still comes out unique once every category has been combined and restricted to
the Permian Basin.

### Why both the `.shp` and the `.csv` are used for each site-type category

The companion `.csv` for each category is convenient because it already carries every attribute
column in one place — but its polygon boundary column is stored as plain WKT text, and for
large/complex polygons that text gets silently truncated at 254 characters (a legacy DBF
character-field limit baked into how these CSVs were exported). The binary `.shp`/`.shx`/`.dbf` files
don't have that limitation — geometry is stored properly there. So this notebook reads **geometry
from the `.shp`**, and only pulls in the handful of extra descriptive columns not present in the
`.shp`'s own attribute table (country, sub-category, per-year detection-count columns) from the
companion `.csv`.

### Connecting the two halves of this notebook: assigning wells to sites

Once the Permian-restricted site catalog exists, the second half of this notebook connects it to the
well-level production data from the previous pipeline stage using a spatial point-in-polygon test:
for every well, check whether its location falls inside one of the site polygons. If it does, the
well is tagged with that site's `site_id`. If a well doesn't fall inside any polygon, it's tagged with
`site_id = -1` — an explicit "not associated with a known site" marker, rather than a missing value,
so it can be filtered on directly downstream.

Because a well's location never changes across its production history, the point-in-polygon test is
run once per **unique well** (not once per well-month row) and the result is broadcast back onto
every row for that well afterward — see Section 7 for why this matters.

### End-to-end flow

```
VIIRS Nightfire zip (.shp + .csv per category)
        │  read geometry from .shp, extra columns from .csv, restrict to Permian bbox
        ▼
permian_flaring_sites.csv                       [saved to permian_only/]
        │
        │  re-read, parse WKT boundaries back into polygons
        ▼
site polygons (in memory)
        │
        │  point-in-polygon test against ...
        │
permian_only/permian_prod_per_well_approx.csv   [from the previous pipeline stage]
        │  parse geometry_wkt back into points, reduce to one row per unique well (api8)
        ▼
site_id assigned per well  ──▶  broadcast back onto every well-month row
        │
        ▼
permian_prod_per_well_with_site_id.csv          [saved to permian_only/]
```

**Scope:** no production values are recalculated or altered anywhere in this notebook — the only
thing added to the well-level table is the `site_id` column.

## 1. Data Source: VIIRS Nightfire Multi-Year Site Catalog

Downloaded from the Colorado School of Mines Earth Observation Group (a free account is required):
[`VNF_multiyear_by_type_2012-2021_v20220822.zip`](https://eogdata.mines.edu/wwwdata/downloads/VNF_multiyear_bubble/VNF_multiyear_by_type_2012-2021_v20220822.zip)

The zip bundles one shapefile (`.shp`/`.shx`/`.dbf`/`.prj`) **and** one companion `.csv` per site
type — `cement`, `coal_mine`, `downstream`, `industrial`, `landfill`, `metallurgy`, `sawmill`,
`unique`, `unknown`, `upstream`, `volcano` — plus two categories (`extinguished_upstream`,
`new_unknown`) that are distributed only as `.kmz` with no attribute table, which are skipped here
since there's no tabular data to read.

Place the downloaded zip anywhere convenient and point `ZIP_PATH` (Section 2) at it. This notebook
reads directly out of the zip in-memory — there's no need to unzip it manually.

## 2. Setup (Imports & Logging)

Shared by both halves of this notebook. `shapely.wkt` is needed later (Sections 6 and 8) to parse
polygon/point boundaries back out of their WKT text representation when reading from CSV.

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────

import zipfile
import logging
import warnings
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely import wkt as shapely_wkt

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

log.info("✓ Imports + logging ready.")


10:01:20 [INFO] ✓ Imports + logging ready.


## 3. Configuration

- `ZIP_PATH` — the VIIRS Nightfire zip downloaded in Section 1.
- `CATEGORIES` — every site type that has a full shapefile + attribute table in the download (the two
  KMZ-only categories are intentionally excluded — see Section 1).
- `PERMIAN_ONLY_FOLDER` — the shared Permian-Basin-scoped output folder used by the previous pipeline
  stage. **Every output from this notebook is written here, as CSV — no Parquet files are produced.**
- `SITES_OUTPUT_CSV` — the flaring-site catalog output (Section 7). It's what Section 8 reads back in.
- `WELLS_CSV_PATH` — the Permian-only well-level production/disposition CSV produced by the previous
  pipeline stage (not the statewide geoparquet — this notebook reads the same CSV that pipeline stage
  saves to `permian_only`).
- `WELLS_WITH_SITE_OUTPUT` — the final combined output (Section 11), also CSV, with polygon/point
  geometry stored as WKT text in a `geometry_wkt` column (the same convention used everywhere else in
  this pipeline for CSV outputs).
- `NO_MATCH_SITE_ID` — the sentinel value assigned to wells that don't fall inside any site polygon.
- The Permian Basin bounding box, matching the one used throughout this pipeline.

In [2]:
# ── Config — only edit these ──────────────────────────────────────────────────

ZIP_PATH = Path("../../../../data/raw/texas/shapefiles.zip")

CATEGORIES = [
    "cement", "coal_mine", "downstream", "industrial", "landfill",
    "metallurgy", "sawmill", "unique", "unknown", "upstream", "volcano",
]

PERMIAN_ONLY_FOLDER = Path("../../../../data/processed/texas/permian_only")
PERMIAN_ONLY_FOLDER.mkdir(parents=True, exist_ok=True)

SITES_OUTPUT_CSV = PERMIAN_ONLY_FOLDER / "permian_flaring_sites.csv"

WELLS_CSV_PATH = PERMIAN_ONLY_FOLDER / "permian_prod_per_well_approx.csv"
WELLS_WITH_SITE_OUTPUT = PERMIAN_ONLY_FOLDER / "permian_prod_per_well_with_site_id.csv"

NO_MATCH_SITE_ID = -1   # assigned to wells that don't fall inside any site polygon

# Permian Basin bounding box — same region used throughout this pipeline
LAT_MIN, LAT_MAX = 29.462935, 34.021515
LONG_MIN, LONG_MAX = -105.21988, -100.036107

# Extra descriptive columns to pull from each category's .csv (not present in the .shp)
EXTRA_CSV_COLS = [
    "id", "COUNTRY", "NCEI_ISO", "type", "category",
    "ID2015", "ID2016", "ID2017", "ID2018", "ID2019", "ID2020", "ID2021",
]

log.info("Sites CSV output : %s", SITES_OUTPUT_CSV.resolve())
log.info("Wells CSV input  : %s", WELLS_CSV_PATH.resolve())
log.info("Final output     : %s", WELLS_WITH_SITE_OUTPUT.resolve())


10:01:20 [INFO] Sites CSV output : /Users/sabare/Desktop/Projects/Final cleaning/data/processed/texas/permian_only/permian_flaring_sites.csv
10:01:20 [INFO] Wells CSV input  : /Users/sabare/Desktop/Projects/Final cleaning/data/processed/texas/permian_only/permian_prod_per_well_approx.csv
10:01:20 [INFO] Final output     : /Users/sabare/Desktop/Projects/Final cleaning/data/processed/texas/permian_only/permian_prod_per_well_with_site_id.csv


## 4. Locate Files Inside the Zip

The zip's internal folder structure may not exactly match what's assumed here, so files are located
by matching on filename alone (case-insensitive), ignoring whatever folder they're nested in.
Mac-created zips sometimes also contain a parallel `__MACOSX/` metadata tree — that's explicitly
excluded.

In [3]:
def find_entry(zf, filename):
    """Find a file inside the zip by name, regardless of its folder path."""
    matches = [
        n for n in zf.namelist()
        if n.split("/")[-1].lower() == filename.lower() and "__MACOSX" not in n
    ]
    if not matches:
        raise FileNotFoundError(f"'{filename}' not found in {ZIP_PATH.name}")
    return matches[0]


## 5. Load One Category, Restricted to the Permian Basin

For a given site type (e.g. `upstream`):

1. Read the polygon geometry straight from the `.shp` (geopandas automatically pulls in the matching
   `.shx`/`.dbf`/`.prj` from the same zip).
2. Read the companion `.csv` and keep only the extra descriptive columns not already in the `.shp`'s
   attribute table, joining them in on `id`.
3. Restrict to rows whose `lat`/`lon` fall inside the Permian Basin bounding box.

In [4]:
def load_category(zf, category):
    """Load one site-type category, geometry from .shp + extra attributes from .csv, restricted to the Permian Basin."""
    shp_entry = find_entry(zf, f"{category}.shp")
    gdf = gpd.read_file(f"zip://{ZIP_PATH}!{shp_entry}")

    # VIIRS Nightfire products are distributed in plain lat/lon (WGS84).
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    csv_entry = find_entry(zf, f"{category}.csv")
    with zf.open(csv_entry) as f:
        extra = pd.read_csv(f, usecols=lambda c: c in EXTRA_CSV_COLS)

    gdf = gdf.merge(extra, on="id", how="left", validate="one_to_one")

    in_permian = gdf["lat"].between(LAT_MIN, LAT_MAX) & gdf["lon"].between(LONG_MIN, LONG_MAX)
    gdf = gdf[in_permian].copy()

    return gdf


## 6. Run Across Every Category and Combine

Every category is checked against the Permian bounding box, not just `upstream` — a handful of sites
classified under `downstream`, `industrial`, or `unknown` can still fall geographically inside the
region, and skipping those categories would silently drop real sites.

In [5]:
all_sites = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for category in CATEGORIES:
        gdf = load_category(zf, category)
        log.info("%-12s %5d site(s) in the Permian Basin", category, len(gdf))
        if not gdf.empty:
            all_sites.append(gdf)

permian_sites = pd.concat(all_sites, ignore_index=True)
permian_sites = gpd.GeoDataFrame(permian_sites, geometry="geometry", crs="EPSG:4326")

print("\nTotal Permian sites:", len(permian_sites))
permian_sites.head()


10:01:21 [INFO] cement           0 site(s) in the Permian Basin
10:01:21 [INFO] coal_mine        0 site(s) in the Permian Basin
10:01:21 [INFO] downstream       1 site(s) in the Permian Basin
10:01:21 [INFO] industrial       2 site(s) in the Permian Basin
10:01:21 [INFO] landfill         0 site(s) in the Permian Basin
10:01:21 [INFO] metallurgy       0 site(s) in the Permian Basin
10:01:21 [INFO] sawmill          0 site(s) in the Permian Basin
10:01:21 [INFO] unique           0 site(s) in the Permian Basin
10:01:21 [INFO] unknown          2 site(s) in the Permian Basin
10:01:21 [INFO] upstream      2323 site(s) in the Permian Basin
10:01:21 [INFO] volcano          0 site(s) in the Permian Basin

Total Permian sites: 2328


,index,id,area,lat,lon,dist_std,cov1,N_dtct,N_dates,T_mean,...,ID2016,ID2017,ID2018,ID2019,ID2020,ID2021,COUNTRY,NCEI_ISO,type,category
0,7413,"IR_2012-2020_tile_-105,30,-90,45_feature_315_s...",3.480373e+06,32.850001,-104.392990,191.787421,0.875021,2483,2225,1598.626459,...,1941.0,2072.0,1662.0,1052.0,1416.0,1226.0,United States,USA,downstream,refinery
1,7182,"IR_2012-2020_tile_-105,30,-90,45_feature_692_s...",2.770297e+06,32.279548,-101.408124,150.389358,0.795641,514,506,1008.042553,...,NaN,NaN,NaN,NaN,NaN,NaN,United States,USA,industrial,chemical
2,9503,"IR_2012-2020_tile_-105,30,-90,45_feature_1555_...",6.910740e+06,31.582207,-102.961337,181.636240,1.274272,89,89,913.454545,...,NaN,NaN,NaN,NaN,NaN,NaN,United States,USA,industrial,powerplant
3,8820,"IR_2012-2020_tile_-105,30,-90,45_feature_488_s...",5.390577e+06,32.494158,-100.409048,163.267525,1.093391,136,130,1914.304348,...,NaN,NaN,NaN,NaN,NaN,NaN,United States,USA,unknown,NaN
4,9162,"IR_2012-2020_tile_-105,30,-90,45_feature_1844_...",6.000643e+06,31.214379,-103.897316,260.143902,1.250769,362,347,1821.668990,...,2220.0,2414.0,1991.0,NaN,NaN,NaN,United States,USA,unknown,oil


## 7. Tidy Columns, Sanity-Check, and Save the Site Catalog

Columns are renamed to a consistent lowercase style, then two checks confirm the result is
well-formed before saving: every `site_id` should be unique (see Section 0's note on how it's
derived), and every geometry should be a valid polygon.

Saved as CSV to `permian_only/` — the format Section 8 reads back in, and the only format this
notebook produces. It's kept identical in structure to how a flaring-site catalog CSV has always been
produced by this pipeline (including writing the DataFrame's row index as the CSV's leading column,
which is why you'll see an unlabeled first column when opening this file — that's expected, not a
bug).

In [6]:
RENAME_MAP = {
    "index": "site_id",
    "id": "catalog_id",
    "COUNTRY": "country",
    "NCEI_ISO": "iso",
    "lat": "latitude",
    "lon": "longitude",
    "N_dtct": "n_dtct",
    "N_dates": "n_dates",
    "T_mean": "t_mean",
    "T_std": "t_std",
    "ID2015": "id2015", "ID2016": "id2016", "ID2017": "id2017", "ID2018": "id2018",
    "ID2019": "id2019", "ID2020": "id2020", "ID2021": "id2021",
}

permian_sites = permian_sites.rename(columns=RENAME_MAP)

FINAL_COLS = [
    "site_id", "catalog_id", "type", "category", "country", "iso",
    "latitude", "longitude", "area", "dist_std", "cov1",
    "n_dtct", "n_dates", "t_mean", "t_std", "time_flag",
    "id2015", "id2016", "id2017", "id2018", "id2019", "id2020", "id2021",
    "geometry",
]
permian_sites = permian_sites[[c for c in FINAL_COLS if c in permian_sites.columns]]

permian_sites.head()


,site_id,catalog_id,type,category,country,iso,latitude,longitude,area,dist_std,...,t_std,time_flag,id2015,id2016,id2017,id2018,id2019,id2020,id2021,geometry
0,7413,"IR_2012-2020_tile_-105,30,-90,45_feature_315_s...",downstream,refinery,United States,USA,32.850001,-104.392990,3.480373e+06,191.787421,...,253.602957,0,6189.0,1941.0,2072.0,1662.0,1052.0,1416.0,1226.0,"POLYGON ((-104.39567 32.84167, -104.39567 32.8..."
1,7182,"IR_2012-2020_tile_-105,30,-90,45_feature_692_s...",industrial,chemical,United States,USA,32.279548,-101.408124,2.770297e+06,150.389358,...,315.158113,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-101.41129 32.27227, -101.41129 32.2..."
2,9503,"IR_2012-2020_tile_-105,30,-90,45_feature_1555_...",industrial,powerplant,United States,USA,31.582207,-102.961337,6.910740e+06,181.636240,...,233.484811,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-102.96097 31.57077, -102.96097 31.5..."
3,8820,"IR_2012-2020_tile_-105,30,-90,45_feature_488_s...",unknown,NaN,United States,USA,32.494158,-100.409048,5.390577e+06,163.267525,...,146.940815,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-100.4123 32.48394, -100.4123 32.484..."
4,9162,"IR_2012-2020_tile_-105,30,-90,45_feature_1844_...",unknown,oil,United States,USA,31.214379,-103.897316,6.000643e+06,260.143902,...,128.963133,1,NaN,2220.0,2414.0,1991.0,NaN,NaN,NaN,"POLYGON ((-103.90157 31.20266, -103.90157 31.2..."


In [7]:
dup_ids = permian_sites["site_id"].duplicated().sum()
print(f"Duplicate site_id values: {dup_ids}")

valid_geom = permian_sites.geometry.is_valid.mean()
print(f"Valid polygon geometries: {valid_geom:.2%}")

print("\nSites per type:")
print(permian_sites["type"].value_counts(dropna=False))


Duplicate site_id values: 0
Valid polygon geometries: 100.00%

Sites per type:
type
upstream      2323
industrial       2
unknown          2
downstream       1
Name: count, dtype: int64


In [8]:
permian_sites.to_csv(SITES_OUTPUT_CSV)

log.info("✓ Saved %s (%.1f MB)", SITES_OUTPUT_CSV.name, SITES_OUTPUT_CSV.stat().st_size / 1_048_576)
print("\nFinal columns:", permian_sites.columns.tolist())
print("Final shape  :", permian_sites.shape)


10:01:21 [INFO] ✓ Saved permian_flaring_sites.csv (5.8 MB)

Final columns: ['site_id', 'catalog_id', 'type', 'category', 'country', 'iso', 'latitude', 'longitude', 'area', 'dist_std', 'cov1', 'n_dtct', 'n_dates', 't_mean', 't_std', 'time_flag', 'id2015', 'id2016', 'id2017', 'id2018', 'id2019', 'id2020', 'id2021', 'geometry']
Final shape  : (2328, 24)


## 8. Reload the Site Catalog and Rebuild Polygon Geometry

Reads the CSV just saved in Section 7 back in (rather than reusing the in-memory `permian_sites`
variable), and reconstructs real polygon geometry from its WKT text column. Reading it back from disk
means this half of the notebook can also be re-run on its own, any time, against whatever the current
`permian_flaring_sites.csv` on disk happens to be — without needing to re-run the extraction in
Sections 4–7 first.

In [9]:
sites_raw = pd.read_csv(SITES_OUTPUT_CSV)

sites_geom_col_candidates = ["geometry_wkt", "geometry"]
sites_geom_col = next((c for c in sites_geom_col_candidates if c in sites_raw.columns), None)
if sites_geom_col is None:
    raise ValueError(
        f"None of {sites_geom_col_candidates} found in {SITES_OUTPUT_CSV.name}. "
        f"Available columns: {sites_raw.columns.tolist()}"
    )
log.info("Using '%s' as the site polygon WKT column.", sites_geom_col)

sites = gpd.GeoDataFrame(
    sites_raw.drop(columns=[sites_geom_col]),
    geometry=sites_raw[sites_geom_col].apply(shapely_wkt.loads),
    crs="EPSG:4326",
)

print("Site polygons loaded:", len(sites))
print("site_id unique:", sites["site_id"].is_unique)
sites[["site_id", "type", "category", "geometry"]].head()


10:01:21 [INFO] Using 'geometry' as the site polygon WKT column.
Site polygons loaded: 2328
site_id unique: True


,site_id,type,category,geometry
0,7413,downstream,refinery,"POLYGON ((-104.39567 32.84167, -104.39567 32.8..."
1,7182,industrial,chemical,"POLYGON ((-101.41129 32.27227, -101.41129 32.2..."
2,9503,industrial,powerplant,"POLYGON ((-102.96097 31.57077, -102.96097 31.5..."
3,8820,unknown,NaN,"POLYGON ((-100.4123 32.48394, -100.4123 32.484..."
4,9162,unknown,oil,"POLYGON ((-103.90157 31.20266, -103.90157 31.2..."


## 9. Load Permian Wells and Reduce to One Row per Unique Well

Reads the Permian-only well-level production/disposition CSV produced by the previous pipeline
stage. Since it's a CSV, its point geometry is stored as WKT text (`geometry_wkt`), so it's parsed
back into real point geometry the same way the site polygons were in Section 8.

The production table has one row per well *per month*, but a well's location — and therefore which
site polygon it falls inside, if any — never changes across those rows. Running the spatial join once
per unique well (identified by `api8` + its point location) rather than once per well-month row avoids
repeating the same geometric test dozens of times over for a well with a long production history; the
result is simply broadcast back onto the full table afterward in Section 11.

In [10]:
wells_raw = pd.read_csv(WELLS_CSV_PATH)

wells = gpd.GeoDataFrame(
    wells_raw.drop(columns=["geometry_wkt"]),
    geometry=wells_raw["geometry_wkt"].apply(shapely_wkt.loads),
    crs="EPSG:4326",
)

print("Well-month rows:", len(wells))
print("Unique wells (api8):", wells["api8"].nunique())

unique_wells = wells.drop_duplicates(subset="api8", keep="first")[["api8", "longitude", "latitude", "geometry"]].copy()
unique_wells = gpd.GeoDataFrame(unique_wells, geometry="geometry", crs=wells.crs)

print("Rows carried into the spatial join:", len(unique_wells))


Well-month rows: 32014046
Unique wells (api8): 222413
Rows carried into the spatial join: 222413


## 10. Point-in-Polygon Test

Each unique well is checked against every site polygon with `predicate="within"` — a well only gets a
match if its point genuinely falls inside a site's boundary, not merely nearby. Site polygons
occasionally overlap slightly, so a small number of wells can land inside more than one; where that
happens, only the first match is kept so the result still has exactly one row per well. Wells matching
no polygon at all get `site_id = -1` (`NO_MATCH_SITE_ID`).

In [11]:
unique_wells = unique_wells.reset_index(drop=True)
unique_wells["_well_row"] = unique_wells.index  # temporary row id, used to catch/collapse duplicate matches

matched = gpd.sjoin(
    unique_wells,
    sites[["site_id", "geometry"]],
    how="left",
    predicate="within",
)

dup_wells = matched["_well_row"].duplicated().sum()
if dup_wells:
    log.warning("%d well(s) fell inside more than one overlapping site polygon — keeping the first match.", dup_wells)
matched = matched.drop_duplicates(subset="_well_row", keep="first")

matched = matched.drop(columns=["index_right", "_well_row"], errors="ignore")
matched["site_id"] = matched["site_id"].fillna(NO_MATCH_SITE_ID).astype(int)

assert len(matched) == len(unique_wells), "row count should match the number of unique wells"

n_matched = (matched["site_id"] != NO_MATCH_SITE_ID).sum()
print(f"Wells matched to a flaring site: {n_matched:,} / {len(matched):,} ({n_matched / len(matched):.2%})")
matched[["api8", "site_id"]].head(10)


Wells matched to a flaring site: 34,481 / 222,413 (15.50%)


,api8,site_id
0,47530372,-1
1,22732018,-1
2,22732262,-1
3,15132344,-1
4,339017,-1
5,23532943,-1
6,17331418,-1
7,47531798,-1
8,47536782,9511
9,47531835,9523


## 11. Broadcast `site_id` Back onto Every Well-Month Row and Save

The per-well `site_id` lookup is merged back onto the full production table by `api8`, so every
well-month row inherits the `site_id` of its well. The final result is saved as CSV to the same
`permian_only` folder as every other output from this notebook. Since CSV can't store geometry
objects natively, point geometry is converted to WKT text in a `geometry_wkt` column first — the same
convention already used for `permian_prod_per_well_approx.csv` (Section 9) and the site catalog
(Section 7).

In [12]:
well_site_lookup = matched[["api8", "site_id"]]

wells_with_site = wells.merge(well_site_lookup, on="api8", how="left")
wells_with_site = gpd.GeoDataFrame(wells_with_site, geometry="geometry", crs=wells.crs)

assert len(wells_with_site) == len(wells), "row count should be unchanged after adding site_id"
assert wells_with_site["site_id"].isna().sum() == 0, "every well should have resolved to either a real site_id or -1"

print("Well-month rows with a flaring site match:", (wells_with_site["site_id"] != NO_MATCH_SITE_ID).sum())
print("Well-month rows with no match (site_id = -1):", (wells_with_site["site_id"] == NO_MATCH_SITE_ID).sum())
wells_with_site.head()


Well-month rows with a flaring site match: 4724748
Well-month rows with no match (site_id = -1): 27289298


,operator_no,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,oil_circulating_bbl,oil_lost_stolen_bbl,oil_bsw_repressure_bbl,oil_legacy_bbl,oil_skimmed_bbl,...,oil_sold_total_bbl,total_vented_flared_mcf,date,lease_key,api8,longitude,latitude,n_wells_with_coordinates,geometry,site_id
0,885555,0.0,161.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,161.0,0.0,2012-04-01,O_10_22061,47530372,-102.923316,31.596287,1.0,POINT (-102.92332 31.59629),-1
1,885555,0.0,140.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,140.0,0.0,2012-09-01,O_10_22061,47530372,-102.923316,31.596287,1.0,POINT (-102.92332 31.59629),-1
2,885555,0.0,150.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,150.0,0.0,2012-11-01,O_10_22061,47530372,-102.923316,31.596287,1.0,POINT (-102.92332 31.59629),-1
3,829482,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.0,0.0,2012-02-01,O_10_25715,22732018,-101.625771,32.433690,2.0,POINT (-101.62577 32.43369),-1
4,829482,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.0,0.0,2012-02-01,O_10_25715,22732262,-101.621242,32.430529,2.0,POINT (-101.62124 32.43053),-1


In [13]:
wells_with_site["geometry_wkt"] = wells_with_site.geometry.to_wkt()

wells_with_site.drop(columns="geometry").to_csv(WELLS_WITH_SITE_OUTPUT, index=False)

log.info("✓ Saved %s (%.1f MB)", WELLS_WITH_SITE_OUTPUT.name, WELLS_WITH_SITE_OUTPUT.stat().st_size / 1_048_576)
print("\nFinal columns:", wells_with_site.columns.tolist())
print("Final shape  :", wells_with_site.shape)


10:07:19 [INFO] ✓ Saved permian_prod_per_well_with_site_id.csv (7409.4 MB)

Final columns: ['operator_no', 'oil_pipeline_bbl', 'oil_truck_bbl', 'oil_tankcar_bbl', 'oil_tank_cleaning_bbl', 'oil_circulating_bbl', 'oil_lost_stolen_bbl', 'oil_bsw_repressure_bbl', 'oil_legacy_bbl', 'oil_skimmed_bbl', 'oil_scrubber_bbl', 'oil_no_disp_code_bbl', 'csgd_field_ops_fuel_mcf', 'csgd_transmission_mcf', 'csgd_processing_plant_mcf', 'csgd_vented_flared_mcf', 'csgd_gas_lift_mcf', 'csgd_repressure_mcf', 'csgd_carbon_black_mcf', 'csgd_underground_storage_mcf', 'csgd_no_disp_code_mcf', 'oil_sold_total_bbl', 'total_vented_flared_mcf', 'date', 'lease_key', 'api8', 'longitude', 'latitude', 'n_wells_with_coordinates', 'geometry', 'site_id', 'geometry_wkt']
Final shape  : (32014046, 32)


## 12. Summary

This notebook produced two files, both CSV, both in `permian_only/`:

| File | Grain | Description |
|---|---|---|
| `permian_flaring_sites.csv` | one row per flaring/combustion site | Every VIIRS Nightfire site polygon within the Permian Basin bounding box, across all site-type categories |
| `permian_prod_per_well_with_site_id.csv` | one row per well per month | The Permian well-level production/disposition table, unchanged, with one new column added |

The flaring site catalog carries, per site:

- `site_id` — a stable numeric identifier, taken directly from the source catalog's own row position
  for that site within its category
- `catalog_id` — the source catalog's full descriptive string ID
- `type` / `category` — what kind of combustion site it is
- `latitude` / `longitude` — the site's representative point location
- detection statistics (`n_dtct`, `n_dates`, `t_mean`, `t_std`, `area`, …) and per-year detection-count
  columns (`id2015`–`id2021`)
- `geometry` — the site's full polygon boundary, read from the shapefile (not the truncated CSV text),
  stored as WKT text in this CSV

The well-level table carries one new column:

- `site_id` — the flaring/combustion site whose polygon the well's location falls inside, or `-1` if
  it doesn't fall inside any known site polygon.

Because `site_id` depends only on a well's fixed location, every well-month row for the same well
carries the same `site_id`.